[Back to Computer Organization and Architecture guideline](Computer-Organization.html)


## **Computer Abstraction and Performance** {#computer-abstraction-and-performance}

A line of source code can look almost effortless: add two values, update a counter, or display a result. The physical machine, however, must represent every value as bits, locate an instruction, move operands through circuits, produce a result, and preserve the next observable state. **Computer architecture and organization** provide the abstractions that make this translation manageable.

This chapter develops two complementary habits. The first is to move confidently between abstraction layers without confusing a software-visible promise with one particular hardware implementation. The second is to evaluate a machine quantitatively rather than treating clock frequency, core count, or transistor count as a complete measure of performance.

The central questions are:

1. What behavior does hardware promise to software?
2. Which internal components cooperate to provide that behavior?
3. How does one instruction move from memory to an architectural result?
4. Which measurements allow two designs to be compared fairly?
5. Why can no design maximize performance, power efficiency, capacity, cost, and reliability at the same time?

The running instruction <code>ADD x5, x1, x2</code> will connect these questions. At the architecture level it promises that register <code>x5</code> receives the sum of <code>x1</code> and <code>x2</code>. At the organization level, a particular processor must fetch its encoding, decode the register numbers, route values into an arithmetic unit, and commit the result.


### **What Are Computer Architecture and Computer Organization?** {#architecture-and-organization}

**Computer architecture** describes the behavior visible at a hardware-software boundary. A useful analogy is a building specification: it defines the rooms, entrances, load limits, and services that occupants may rely on. **Computer organization** is the engineering plan that satisfies that specification: where beams are placed, how pipes are routed, and which materials are selected.

For a processor, the most important architectural contract is the **instruction set architecture (ISA)**. It specifies instructions, registers, data representations, addressing behavior, and the effects that programs can observe. Organization, often called **microarchitecture** at the processor level, chooses structures such as pipelines, caches, execution units, and control logic.

![Architecture defines software-visible behavior, while organization chooses an implementation that realizes the same ISA contract.](assets/architecture-organization-isa.svg){fig-align="center" width="100%"}

Consider <code>ADD x5, x1, x2</code>. Its architectural meaning is approximately

$$
R[5] \leftarrow \bigl(R[1] + R[2]\bigr) \bmod 2^w,
$$

where $R[i]$ means register $i$ and $w$ is the architectural register width. The modulo operation means only the low $w$ bits remain after fixed-width addition. Software can depend on this result. Software normally cannot depend on whether the processor used a ripple-carry adder or a carry-lookahead adder, whether the instruction waited in a queue, or whether another instruction executed at the same time. Those are organizational choices.

The distinction is practical rather than merely terminological:

| Layer | Main question | Examples | Can ordinary program behavior depend on it? |
|---|---|---|---|
| Architecture / ISA | What does the machine promise? | instructions, registers, address space, exception semantics | Yes |
| Microarchitecture / organization | How is the promise implemented? | pipeline, cache, branch predictor, execution units | Normally no, except through timing and other non-functional effects |
| Circuit and physical implementation | How is the organization realized electronically? | gates, transistor technology, voltage, layout | Normally no, except through performance, energy, reliability, and leakage |

This separation solves two problems. Software can survive hardware replacement when the new processor implements the same ISA, and hardware designers can improve the organization without requiring every program to be rewritten. It also explains why “architecture” is sometimes used broadly for a complete system design while “ISA” is used for the precise software-visible processor contract.

A change is architectural when software must know about it to remain correct. Adding a new instruction, changing the width of an address, or changing exception behavior is architectural. Increasing cache capacity, replacing an in-order pipeline with an out-of-order engine, or changing circuit technology is organizational as long as the original observable behavior remains valid.


### **Layers of Abstraction** {#layers-of-abstraction}

An **abstraction** hides lower-level detail behind a smaller interface. It is not a claim that the hidden detail is unimportant. It is a way to reason locally. A programmer can call a function without tracking individual transistors; a processor designer can implement an instruction without knowing which application will eventually execute it.

![A traditional view of computer abstraction layers, from hardware and firmware to software.](assets/computer-abstraction-layers.svg){fig-align="center" width="42%"}

*Image source: [Computer abstraction layers](https://commons.wikimedia.org/wiki/File:Computer_abstraction_layers.svg), Miko3k and Tene, CC BY-SA 3.0 / GFDL.*

No single stack diagram captures every modern system, but the following responsibilities are useful:

| Layer | Representative artifact | Responsibility hidden from the layer above |
|---|---|---|
| Application | browser, database, scientific program | algorithms, user workflow, domain meaning |
| High-level language and libraries | Python, C, runtime library | portable data types, functions, memory conventions |
| Operating system | processes, virtual memory, files, device APIs | protection, resource sharing, device management |
| ISA and machine code | RISC-V instructions and registers | processor instruction semantics |
| Microarchitecture | datapath, control, pipeline, cache | cycles, internal queues, speculation |
| Digital logic | gates, registers, memories | Boolean and sequential circuit behavior |
| Devices and physical implementation | transistors, wires, storage cells | electrical switching and fabrication constraints |

The boundaries are crossed continually. A library call may request an operating-system service; an exception transfers control from an application to privileged software; a cache miss forces the microarchitecture to wait for a lower memory level. Good abstraction therefore means **controlled interaction**, not complete isolation.

#### **Application to Machine Code** {#application-to-machine-code}

Suppose a C function returns the sum of two integer arguments. Source code states the intended computation but does not name physical gates or instruction bits. A compiler selects instructions for a target ISA, an assembler encodes those instructions, a linker combines code and resolves symbols, and an operating-system loader places the executable into memory.

<details>
<summary>One operation across C, RISC-V assembly, and machine code</summary>

~~~c
int add_two(int a, int b) {
    return a + b;
}
~~~

Under a common RISC-V calling convention, the arguments and return value use registers <code>a0</code> and <code>a1</code>:

~~~asm
add a0, a0, a1    # a0 receives a0 + a1
ret                # return to the address in ra
~~~

One valid 32-bit encoding of those instructions is:

~~~text
00 b5 05 33    add a0, a0, a1    (instruction word 0x00b50533)
00 00 80 67    jalr x0, 0(ra)     (the instruction represented by ret)
~~~

</details>

The precise output can change with compiler version, optimization settings, ABI, and target extensions. That variation is acceptable because the source language and ISA define behavior, not one mandatory instruction sequence. This is an important optimization freedom: two compiled programs may contain different instructions yet be equivalent at the source level.

The translation path is therefore not “English-like code becomes electricity” in one step. It is a chain of contracts:

$$
\text{source semantics}
\rightarrow \text{compiler IR}
\rightarrow \text{assembly}
\rightarrow \text{machine encoding}
\rightarrow \text{ISA effects}
\rightarrow \text{circuit transitions}.
$$

Each arrow preserves the behavior required by the layer to its left while introducing details needed by the layer to its right.

#### **The ISA as a Hardware-Software Contract** {#isa-contract}

An ISA usually defines more than an instruction-name list. Its contract includes:

- **machine state:** general registers, special registers, program counter, and privilege state;
- **instruction semantics:** which state is read, which operation occurs, and which state is updated;
- **instruction encodings:** how opcodes, register numbers, and immediate values occupy bits;
- **memory behavior:** address size, alignment requirements, ordering rules, and atomic operations;
- **exceptions and interrupts:** which events transfer control and what state is preserved;
- **privileged mechanisms:** the operations available only to an operating system or monitor.

The ISA deliberately leaves many questions unanswered. It does not normally specify cache capacity, pipeline depth, branch-predictor design, physical register count, or exact instruction latency. Leaving these details open enables multiple implementations to compete on speed, power, area, and cost while executing the same binaries.

| Software may rely on | Software should not normally assume |
|---|---|
| the result written by an instruction | which internal ALU produced it |
| the defined ordering of visible memory effects | whether a cache hit occurred |
| documented exceptions and privilege behavior | the number of pipeline stages |
| architectural register and address widths | the physical number of registers or buffers |

This contract view provides a test for later chapters: when a mechanism is introduced, ask whether it changes architectural meaning or only changes how efficiently that meaning is produced.


### **The Stored-Program Computer** {#stored-program-computer}

A **stored-program computer** represents instructions in addressable memory rather than fixing the entire computation in wiring. Changing a program can therefore mean loading different instruction bits, not rebuilding the machine. Data and instructions share the same general storage model, allowing one processor to perform many tasks.

The classic idea is associated with the von Neumann model. Modern processors often use separate instruction and data caches internally, which resembles a Harvard organization, while still presenting a largely unified architectural address space. This is a useful reminder that one architecture can be implemented by a more specialized organization.

![A processor, shared instruction/data memory, and I/O devices exchange addresses, data, and control information.](assets/stored-program-system.svg){fig-align="center" width="100%"}

#### **Processor, Memory, and I/O** {#processor-memory-io}

At the highest useful level, a general-purpose computer contains three cooperating subsystems:

1. The **processor** reads instructions and transforms architectural state. Its control logic coordinates operations, its arithmetic logic unit performs calculations and comparisons, its register file holds fast operands, and its program counter identifies the next instruction.
2. **Memory** stores instruction encodings and program data. It is addressable: a numeric address selects a location or block. Main memory is larger but much slower than processor registers, so later chapters introduce caches and a hierarchy.
3. **I/O devices** connect computation to the external world. Displays, networks, keyboards, sensors, and storage devices operate at different rates and usually communicate through device controllers.

For the running instruction <code>ADD x5, x1, x2</code>, instruction memory provides the encoded operation, the register file provides two values, the ALU produces their sum, and the register file receives the result. A load instruction adds a data-memory read; a store adds a data-memory write; an I/O instruction or memory-mapped access may reach a device controller.

These components exist because no single storage or computational structure can be simultaneously fastest, largest, cheapest, and lowest-power. Registers are close to execution units but scarce. Main memory has much greater capacity but requires longer communication. Storage persists across power loss but is slower again. Organization is largely the art of making those unequal components cooperate.

#### **Buses and Interconnects** {#buses-and-interconnects}

An **interconnect** transports information between components. A traditional bus can be understood as three logical responsibilities:

- **address information** identifies a destination or storage location;
- **data information** carries an instruction, operand, or result;
- **control information** identifies the operation, size, direction, status, and timing.

A memory read transaction illustrates the separation. The processor presents an address and a read request. Memory or a cache determines whether it owns that address, retrieves the requested bytes, and returns both data and a completion response. A write transaction sends an address and data together. Arbitration is required if several agents request a shared resource.

Two measurements should not be confused:

$$
\text{bandwidth} = \frac{\text{bytes transferred}}{\text{elapsed time}},
$$

while **latency** is the time between issuing one request and receiving the required response. A useful first-order transfer model is

$$
T_{\text{transfer}} \approx T_{\text{fixed}} + \frac{B}{R},
$$

where $T_{\text{fixed}}$ is setup and response latency, $B$ is the number of bytes, and $R$ is sustained bandwidth in bytes per second. Small dependent reads are often latency-sensitive because the fixed term dominates. Large streaming transfers are often bandwidth-sensitive because $B/R$ dominates.

| Interconnect style | Strength | Limitation | Typical use |
|---|---|---|---|
| Shared bus | simple and inexpensive | agents contend for one shared path | small or low-bandwidth systems |
| Point-to-point link | high dedicated bandwidth | many components require many links or switches | processor-memory and peripheral links |
| Switched network / network-on-chip | scalable concurrent routes | routing, buffering, and coherence become complex | multicore systems and large chips |

The word “bus” remains useful conceptually even when a modern system uses packets and switched links: every transfer still needs a destination, payload, operation, and completion rule.


### **The Instruction Execution Cycle** {#instruction-execution-cycle}

The **instruction execution cycle** is a conceptual decomposition of the work required to transform one architectural state into the next. It is often summarized as fetch, decode, and execute, but separating memory access and write-back makes data movement and state commitment easier to see.

~~~text
INSTRUCTION-CYCLE
    instruction_address <- PC
    instruction <- memory[instruction_address]       // fetch
    PC <- address of the next sequential instruction
    operation, operands <- decode(instruction)       // decode
    result, address, branch <- perform(operation)    // execute
    data <- optional memory read or write            // memory
    update architectural register or next PC         // write-back
~~~

The pseudocode expresses dependencies, not a universal timing implementation. A single-cycle processor may perform all required work in one long cycle. A multi-cycle processor may reuse one ALU over several cycles. A pipelined processor may have many instructions occupying different stages simultaneously.

![The conceptual instruction cycle illustrated with ADD x5, x1, x2 and concrete register values.](assets/instruction-cycle.svg){fig-align="center" width="100%"}

#### **Fetch** {#fetch}

During **fetch**, the program counter supplies an instruction address. The memory system returns instruction bits, which are retained for decoding, and the processor forms a candidate next sequential PC. In a fixed-width 32-bit instruction stream, that candidate may be $PC+4$; variable-length ISAs need more elaborate length detection.

Fetch appears simple but already depends on several mechanisms. The address must be valid and executable, virtual-to-physical translation may be required, an instruction cache may hit or miss, and a branch predictor may provide a speculative next PC. These mechanisms should accelerate fetching without changing the final architectural behavior.

#### **Decode** {#decode}

During **decode**, fields in the instruction are interpreted according to the ISA. The opcode identifies an operation class; register fields select operands and destinations; immediate fields contribute constants, displacements, or branch offsets. Control logic then determines which datapath resources are needed.

For <code>ADD x5, x1, x2</code>, decoding identifies an integer addition, source registers 1 and 2, and destination register 5. Reading values from the register file is an organizational operation. The architectural contract only requires the final sum to appear in <code>x5</code> when the instruction completes normally.

#### **Execute** {#execute}

During **execute**, a functional unit performs the operation needed by the instruction. Arithmetic instructions use an ALU; branches compare values and form a target; loads and stores calculate an effective address; multiplication or floating-point instructions may use specialized units.

Execution is therefore broader than “do arithmetic.” It creates the information required for the next stage. For a load such as <code>LOAD x5, 16(x1)</code>, execute may compute

$$
\text{effective address} = R[1] + \operatorname{signExtend}(16),
$$

where $R[1]$ is a base address and sign extension converts the instruction's immediate field to the architectural width while preserving its signed value.

#### **Memory Access and Write-Back** {#memory-access-write-back}

Only instructions that communicate with memory require a data-memory operation. A load retrieves bytes and converts them to an architectural value; a store sends bytes and a write request. Arithmetic instructions usually forward the ALU result without a data-memory access.

**Write-back** makes a result visible in architectural state, commonly by updating a destination register. A branch instead selects a next PC, and a store's architectural effect is the memory write. Real out-of-order processors may calculate results early but delay retirement so exceptions still appear in program order. This distinction between *producing* a value and *committing* architectural state becomes important in later chapters.

The following deliberately small interpreter turns the conceptual stages into executable Python. Its instruction tuples are not real RISC-V encodings; they make the state transitions visible without hiding them in a complete simulator.

<details>
<summary>Python implementation: a tiny stored-program instruction cycle</summary>

~~~python
from dataclasses import dataclass, field
from typing import TypeAlias

Instruction: TypeAlias = tuple[object, ...]


@dataclass
class TinyMachine:
    """A teaching model with registers, data memory, a PC, and five operations."""

    program: list[Instruction]
    registers: list[int] = field(default_factory=lambda: [0] * 8)
    memory: dict[int, int] = field(default_factory=dict)
    pc: int = 0
    halted: bool = False
    trace: list[str] = field(default_factory=list)

    def step(self) -> None:
        if self.halted:
            return
        if not 0 <= self.pc < len(self.program):
            raise IndexError("program counter is outside instruction memory")

        # 1. Fetch: use PC as an instruction-memory index, then advance it.
        instruction_address = self.pc
        instruction = self.program[instruction_address]
        self.pc += 1
        self.trace.append(f"fetch  pc={instruction_address}: {instruction}")

        # 2. Decode: separate the opcode from its operand fields.
        opcode, *operands = instruction
        self.trace.append(f"decode opcode={opcode}, operands={operands}")

        # Execute may produce a pending register write or a memory effect.
        register_write: tuple[int, int] | None = None

        # 3. Execute: choose behavior from the decoded operation.
        if opcode == "LI":
            destination, value = operands
            register_write = (int(destination), int(value))
        elif opcode == "ADD":
            destination, left, right = map(int, operands)
            result = self.registers[left] + self.registers[right]
            register_write = (destination, result)
        elif opcode == "LOAD":
            destination, address = map(int, operands)
            # 4. Memory access: retrieve data at the effective address.
            register_write = (destination, self.memory.get(address, 0))
        elif opcode == "STORE":
            source, address = map(int, operands)
            # 4. Memory access: make the store visible in data memory.
            self.memory[address] = self.registers[source]
        elif opcode == "HALT":
            self.halted = True
        else:
            raise ValueError(f"unknown opcode: {opcode}")

        # 5. Write-back: commit a pending result to architectural state.
        if register_write is not None:
            destination, value = register_write
            self.registers[destination] = value
            self.trace.append(f"write  x{destination}={value}")

    def run(self, maximum_steps: int = 100) -> None:
        for _ in range(maximum_steps):
            if self.halted:
                return
            self.step()
        raise RuntimeError("program did not halt within the step budget")


program: list[Instruction] = [
    ("LI", 1, 7),          # x1 <- 7
    ("LI", 2, 4),          # x2 <- 4
    ("ADD", 5, 1, 2),      # x5 <- x1 + x2
    ("STORE", 5, 0),       # memory[0] <- x5
    ("LOAD", 6, 0),        # x6 <- memory[0]
    ("HALT",),
]

machine = TinyMachine(program)
machine.run()

assert machine.registers[5] == 11
assert machine.memory[0] == 11
assert machine.registers[6] == 11

print("\n".join(machine.trace))
~~~

</details>

The model omits caches, binary encodings, interrupts, privilege, and timing. That omission is intentional: it preserves the architectural state-transition idea while exposing exactly which details later chapters must add.


### **Measuring Performance** {#measuring-performance}

Performance is meaningful only for a stated workload and metric. “Machine A is faster” is incomplete unless it identifies what work was executed, which inputs were used, whether the outputs were equivalent, and whether *faster* means lower response time, higher throughput, or both.

#### **Latency and Throughput** {#latency-and-throughput}

**Latency**, also called response time, is how long one task takes from request to completion:

$$
L = t_{\text{completion}} - t_{\text{arrival}}.
$$

**Throughput** is how much work completes per unit time:

$$
Q = \frac{N_{\text{completed}}}{\Delta t}.
$$

Here $L$ is latency, $t_{\text{arrival}}$ and $t_{\text{completion}}$ bound one request, $Q$ is throughput, $N_{\text{completed}}$ is the number of completed tasks, and $\Delta t$ is the observation interval.

![A four-stage pipeline can improve completion rate without reducing the end-to-end latency of one job.](assets/latency-throughput.svg){fig-align="center" width="96%"}

The two metrics can move differently. A pipelined system may preserve approximately the same latency for one instruction while completing many instructions at overlapping stages. Batching can raise storage or network throughput while making an individual request wait longer. Interactive systems usually emphasize tail latency; servers and accelerators often care strongly about sustained throughput.

| Scenario | Primary concern | Why |
|---|---|---|
| keystroke-to-screen response | latency | the user waits for one response |
| web server under load | throughput and tail latency | many requests must finish without extreme delays |
| scientific batch computation | total elapsed time / throughput | completion of the whole workload matters |
| real-time controller | bounded worst-case latency | a late correct answer may be unusable |

#### **Clock Rate and Cycle Time** {#clock-rate-and-cycle-time}

In a synchronous processor, the clock divides activity into coordinated intervals. **Clock rate** $f_{\text{clock}}$ counts cycles per second, while **cycle time** $T_{\text{cycle}}$ is seconds per cycle:

$$
T_{\text{cycle}} = \frac{1}{f_{\text{clock}}}.
$$

A 3 GHz clock performs $3\times 10^9$ cycles per second, so

$$
T_{\text{cycle}} = \frac{1}{3\times 10^9}\text{ s} \approx 333\text{ ps}.
$$

The cycle must be long enough for the slowest required combinational path, plus register timing margins. Shortening the cycle may require dividing work into more pipeline stages. That can raise clock rate while increasing pipeline overhead, branch penalties, and design complexity.

Clock rate is therefore a machine parameter, not a complete performance result. One architecture may need fewer instructions; one microarchitecture may need fewer cycles per instruction; one workload may spend more time waiting for memory. A 4 GHz processor is not automatically faster than a 3 GHz processor.

#### **Instruction Count, CPI, and CPU Time** {#instruction-count-cpi-cpu-time}

The classic CPU performance equation separates three contributors:

$$
T_{\text{CPU}}
= IC \times CPI \times T_{\text{cycle}}
= \frac{IC \times CPI}{f_{\text{clock}}}.
$$

- $T_{\text{CPU}}$ is time spent executing instructions on the CPU for the measured workload.
- $IC$ is the **dynamic instruction count**, meaning instructions actually executed, including loop repetitions.
- $CPI$ is average **cycles per instruction** for this instruction stream on this organization.
- $T_{\text{cycle}}$ is seconds per cycle.
- $f_{\text{clock}}$ is cycles per second and equals $1/T_{\text{cycle}}$.

The units confirm the formula:

$$
\text{instructions}
\times \frac{\text{cycles}}{\text{instruction}}
\times \frac{\text{seconds}}{\text{cycle}}
= \text{seconds}.
$$

![The CPU performance equation separates instruction count, CPI, and clock period, with a worked two-design comparison.](assets/cpu-performance-equation.svg){fig-align="center" width="100%"}

Average CPI depends on the instruction mix. If class $i$ represents a fraction $F_i$ of executed instructions and takes $CPI_i$ cycles on average, then

$$
CPI_{\text{avg}} = \sum_i F_i \times CPI_i,
\qquad \sum_i F_i = 1.
$$

A class with a high $CPI_i$ matters little when it is rare, while a modest cost matters greatly when it occurs frequently. Cache misses and branch mispredictions often change CPI without changing ISA semantics or instruction count.

<details>
<summary>Python implementation: compare CPU designs with the full performance equation</summary>

~~~python
from dataclasses import dataclass


@dataclass(frozen=True)
class CpuDesign:
    name: str
    instruction_count: float
    average_cpi: float
    clock_rate_hz: float

    def cpu_time_seconds(self) -> float:
        """Return IC x CPI / clock rate, with units of seconds."""
        return self.instruction_count * self.average_cpi / self.clock_rate_hz


def weighted_cpi(instruction_mix: dict[str, tuple[float, float]]) -> float:
    """Combine (frequency, class CPI) pairs into one average CPI."""
    total_frequency = sum(frequency for frequency, _ in instruction_mix.values())
    if abs(total_frequency - 1.0) > 1e-9:
        raise ValueError("instruction frequencies must sum to 1")

    return sum(
        frequency * class_cpi
        for frequency, class_cpi in instruction_mix.values()
    )


mix = {
    "integer ALU": (0.55, 1.0),
    "load": (0.25, 2.0),
    "store": (0.05, 2.0),
    "branch": (0.15, 2.0),
}

assert abs(weighted_cpi(mix) - 1.45) < 1e-9

design_a = CpuDesign("A", 2.0e9, 1.2, 3.0e9)
design_b = CpuDesign("B", 1.6e9, 1.8, 4.0e9)

time_a = design_a.cpu_time_seconds()
time_b = design_b.cpu_time_seconds()
speedup_b_over_a = time_a / time_b

assert abs(time_a - 0.80) < 1e-9
assert abs(time_b - 0.72) < 1e-9
assert abs(speedup_b_over_a - (10 / 9)) < 1e-9

print(f"Design A CPU time: {time_a:.2f} s")
print(f"Design B CPU time: {time_b:.2f} s")
print(f"B is {speedup_b_over_a:.2f}x as fast as A for this workload")
~~~

</details>

CPU time is not always wall-clock time. Elapsed time can include I/O waits, operating-system scheduling, synchronization, and work on accelerators. The equation remains valuable because it explains processor execution, but the measured metric must match the user's actual experience.

#### **Speedup and Amdahl's Law** {#speedup-and-amdahls-law}

When lower execution time is better, the speedup of a new design relative to an old design is

$$
S = \frac{T_{\text{old}}}{T_{\text{new}}}.
$$

A speedup of $2$ means the new execution time is half the old time. Reversing the fraction would report slowdown, so naming the baseline matters.

Optimizing one component cannot accelerate time spent elsewhere. If fraction $p$ of the original execution time is improved by a factor $s$, while fraction $1-p$ is unchanged, Amdahl's law gives

$$
S_{\text{overall}} = \frac{1}{(1-p)+\frac{p}{s}}.
$$

- $p$ is the fraction of original time affected by the improvement.
- $s$ is the local speedup of that affected part.
- $1-p$ is the unchanged fraction and becomes the limiting term.

If 80% of execution is made four times faster, the new normalized time is

$$
(1-0.8)+\frac{0.8}{4}=0.2+0.2=0.4,
$$

so the overall speedup is $1/0.4=2.5$, not $4$. Even an infinitely fast replacement of the improved part cannot exceed $1/(1-0.8)=5$ because the remaining 20% still takes time.

![Amdahl's law shows that a fixed serial or unchanged fraction limits speedup as resources increase.](assets/amdahls-law.svg){fig-align="center" width="70%"}

*Image source: [Amdahl's Law](https://commons.wikimedia.org/wiki/File:AmdahlsLaw.svg), Wikimedia contributors, CC BY-SA 3.0.*

<details>
<summary>Python experiment: local improvement versus overall speedup</summary>

~~~python
def amdahl_speedup(fraction_improved: float, local_speedup: float) -> float:
    """Return 1 / ((1-p) + p/s) after validating the model inputs."""
    if not 0.0 <= fraction_improved <= 1.0:
        raise ValueError("fraction_improved must be between 0 and 1")
    if local_speedup < 1.0:
        raise ValueError("local_speedup must be at least 1")

    unchanged_time = 1.0 - fraction_improved
    improved_time = fraction_improved / local_speedup
    return 1.0 / (unchanged_time + improved_time)


for local_speedup in (1, 2, 4, 8, 1000):
    overall = amdahl_speedup(0.80, local_speedup)
    print(f"local={local_speedup:>4}x -> overall={overall:.3f}x")

assert abs(amdahl_speedup(0.80, 4) - 2.5) < 1e-9
assert amdahl_speedup(0.80, 1000) < 5.0
~~~

</details>

Amdahl's law is not limited to parallel processors. The improved part could be floating-point execution, cache misses, compression, database queries, or hardware acceleration. The discipline is always the same: measure how much original time the optimized component actually consumed.


### **Technology Trends and Design Constraints** {#technology-trends-and-design-constraints}

Architecture is shaped by what technology can deliver economically and reliably. Earlier processor generations often gained performance from higher clock rates and greater transistor budgets. Modern design must distribute that budget across cores, caches, specialized units, interconnects, protection, and power-management structures.

![Power, memory, cost, and reliability constrain every complete computer design.](assets/technology-design-constraints.svg){fig-align="center" width="100%"}

#### **The Power Wall** {#power-wall}

Switching transistors consumes dynamic power. A widely used first-order model is

$$
P_{\text{dynamic}} \approx \alpha C V^2 f,
$$

where $\alpha$ is the fraction of capacitance that switches per cycle, $C$ is effective switched capacitance, $V$ is supply voltage, and $f$ is switching frequency. The formula is approximate, but its intuition is crucial:

- more switching activity or capacitance raises energy use;
- higher frequency creates more switching events per second;
- voltage has a quadratic effect, so reducing it can save substantial power;
- lower voltage may also reduce maximum reliable frequency.

Static leakage, memory, interconnects, and cooling also matter. The **power wall** means frequency cannot be increased indefinitely without exceeding energy and thermal limits. This helps explain the move toward multicore designs, clock gating, dynamic voltage and frequency scaling, and specialized accelerators.

<details>
<summary>Python experiment: normalized dynamic-power tradeoff</summary>

~~~python
def normalized_dynamic_power(
    activity: float,
    capacitance: float,
    voltage: float,
    frequency: float,
) -> float:
    """Return a proportional alpha*C*V^2*f value for design comparison."""
    return activity * capacitance * voltage**2 * frequency


# Hold activity and capacitance constant so voltage/frequency effects are clear.
baseline = normalized_dynamic_power(0.20, 1.0, voltage=1.0, frequency=3.0)
lower_power = normalized_dynamic_power(0.20, 1.0, voltage=0.8, frequency=2.4)
ratio = lower_power / baseline

# Voltage falls to 80% and frequency falls to 80%: 0.8^2 * 0.8 = 0.512.
assert abs(ratio - 0.512) < 1e-12
print(f"new dynamic power is {ratio:.1%} of baseline")
~~~

</details>

The result does not say the second design completes the same workload with the same performance. Power, energy, and execution time must be measured together. Lower instantaneous power can still consume substantial energy if execution lasts much longer.

#### **The Memory Wall** {#memory-wall}

Processor arithmetic can be wasted when required data is not available. The **memory wall** describes the growing importance of the gap between computation and data movement. Two different limitations appear:

- **latency:** how long a dependent request waits before data arrives;
- **bandwidth:** how many bytes can be delivered per second across many requests.

A simple CPI decomposition is

$$
CPI_{\text{effective}}
= CPI_{\text{base}}
+ \text{memory stall cycles per instruction}.
$$

Suppose a core has base CPI 1, encounters 0.05 costly misses per instruction, and each exposed miss adds 40 cycles on average. The stall term is $0.05\times 40=2$, so effective CPI becomes 3. Two thirds of processor cycles are now associated with memory stalls, making a faster ALU almost irrelevant.

Caches exploit locality, prefetchers request data early, multiple outstanding requests overlap latency, and high-bandwidth memory supports streams. None is universally best. A cache helps repeated nearby accesses; prefetching can waste bandwidth when predictions are wrong; parallel requests help only when enough independent work exists.

#### **Cost and Reliability** {#cost-and-reliability}

A design must be manufacturable and dependable, not merely fast in simulation. A rough manufacturing relationship is

$$
\text{cost per good die}
\approx
\frac{\text{wafer cost}}
{\text{dies per wafer}\times\text{yield}},
$$

where **yield** is the fraction of fabricated dies that meet requirements. Larger dies generally reduce the number of dies per wafer and expose each die to more defect opportunities. Packaging, external memory, cooling, and validation also contribute to system cost.

Reliability asks whether a system continues to produce correct results over time and under faults. For independent components that must all work, a simple series model is

$$
R_{\text{system}} = \prod_{i=1}^{n} R_i,
$$

where $R_i$ is the probability that required component $i$ operates correctly over a stated interval. The independence assumption is often imperfect because temperature, power supply, and environmental events can affect several components together, but the product illustrates why adding required components can reduce unprotected system reliability.

Design responses include parity, error-correcting codes, redundant execution, spare capacity, thermal monitoring, and graceful degradation. These mechanisms consume area, energy, or time. Their value depends on the cost of an incorrect or unavailable system: a toy, laptop, data center, and spacecraft have very different reliability contracts.


### **Comparing Computer Designs** {#comparing-computer-designs}

A fair comparison begins with equivalent work and correct output. Comparing processor advertisements by maximum clock rate violates that rule because instruction count, CPI, memory behavior, power limits, and workload all differ. Benchmarking should therefore be treated as a controlled experiment.

A practical workflow is:

1. **Define the workload.** Specify program, input, compiler, ISA target, and required output.
2. **Choose the user-facing metric.** Decide whether latency, throughput, energy, memory capacity, predictability, or cost is primary.
3. **Hold irrelevant conditions constant.** Control software versions, cooling mode, data placement, and measurement interval.
4. **Measure components as well as totals.** Record elapsed time and, when available, instruction count, CPI, cache misses, bandwidth, and power.
5. **Repeat and report variation.** Warm-up effects, background work, frequency scaling, and randomness can distort one run.
6. **Explain the mechanism.** A result is more useful when the observed change can be connected to instructions, cycles, data movement, or resource limits.

| Question | Appropriate measure | Common misleading shortcut |
|---|---|---|
| How long does one task take? | end-to-end latency | clock rate alone |
| How many tasks finish under load? | sustained throughput plus tail latency | best single-request result |
| How efficiently does the CPU execute? | $IC$, average CPI, clock rate, CPU time | CPI or instruction count in isolation |
| How much useful work fits an energy budget? | energy per task or performance per watt | peak performance at unconstrained power |
| Will deadlines always be met? | bounded or high-percentile latency | average latency |
| Is one optimization worth implementing? | measured affected fraction and Amdahl speedup | local component speedup |

Different computers can therefore be best for different contracts:

| Design context | Likely priorities | Acceptable tradeoff |
|---|---|---|
| interactive laptop | responsive latency, battery life, quiet cooling | lower sustained peak performance |
| data-center service | throughput, tail latency, performance per watt, availability | higher design and infrastructure complexity |
| embedded controller | predictable timing, low power, low cost | smaller memory and lower general-purpose throughput |
| scientific accelerator | arithmetic and memory throughput for a narrow workload | reduced flexibility and explicit data movement |

**Chapter summary.** Architecture states what software can observe; organization chooses how hardware provides it. Abstraction layers permit local reasoning while preserving explicit contracts. The stored-program model connects processor, memory, and I/O through controlled transfers. The instruction cycle explains how one encoded operation becomes an architectural state update. Performance must be decomposed into workload, instruction count, CPI, clock period, latency, and throughput. Amdahl's law prevents local improvements from being mistaken for whole-system speedup. Finally, power, memory, cost, and reliability ensure that the “fastest” design is not automatically the most useful design.
